# 🏏 Godrej Cricket Intelligence — In-Play Win Probability Modeling
### Predictive Data Science & Machine Learning for Real-Time T20 Analytics
**Team Elite Crew** | College Project for Godrej Data Analysis Training Program

---
## 1. Problem Formulation & Data Science Objective
In limited-overs T20 cricket, estimating the **in-play victory probability** $P(\text{Win} = 1 \mid \text{State}_t)$ enables real-time broadcast insights, odds estimation, and tactical decision-making.

Unlike standard classification problems, cricket chase dynamics operate under strict physical laws and non-linear pressure dynamics:
- **Physical Boundary Invariants:** Reaching the target ($runs \ge target$) is a deterministic $100\%$ win. Losing 10 wickets or exhausting deliveries without reaching the target is a deterministic $0\%$ win.
- **Resource Scarcity & Non-Linearity:** Required Run Rate (RRR) must be balanced against remaining wickets via a **Pressure Index** ($PI = \frac{\text{RRR}}{\text{Wickets in Hand} + 1}$).

In [ ]:
import os
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import joblib

from src.ml_win_probability import CricketWinProbabilityModel, WinProbabilityState
from src.train_ml_model import train_and_evaluate, generate_synthetic_t20_corpus
print("✓ Successfully imported Godrej Cricket Intelligence ML Engine.")

## 2. Model Training & Statistical Calibration
We train an L2-penalized **Logistic Regression** model on a calibrated T20 transition state corpus and evaluate it across:
1. **Classification Accuracy (89.0%)**
2. **Area Under the ROC Curve (ROC-AUC: 0.9557)**
3. **Brier Calibration Loss (0.0766)** ($< 0.15$ indicates well-calibrated probabilities)
4. **Log-Loss (0.2525)**

In [ ]:
# Load evaluation metadata and print model metrics
import json
metrics_path = os.path.join("..", "models", "model_evaluation_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        meta = json.load(f)
    print("Model Type:", meta["model_type"])
    print("Test Accuracy:", f"{meta['metrics']['test_accuracy']*100:.2f}%")
    print("Test ROC-AUC: ", meta["metrics"]["test_roc_auc"])
    print("Test Brier:   ", meta["metrics"]["test_brier_score"])
    print("Test LogLoss: ", meta["metrics"]["test_log_loss"])
    print("\nFeature Coefficients:")
    for k, v in meta["feature_coefficients"].items():
        print(f"  {k:<20}: {v:+.4f}")

## 3. In-Play Scenario Inference
Testing the model across match situations for Chennai Super Kings vs Gujarat Titans (Target: 215 runs).

In [ ]:
model = CricketWinProbabilityModel()

scenarios = [
    ("Start of Chase", 0, 0, 0.0, 215),
    ("Powerplay End", 52, 1, 6.0, 215),
    ("Middle Overs (99/3)", 99, 3, 6.6, 215),
    ("Death Overs Pressure", 160, 6, 16.0, 215),
    ("Final Over Drama", 205, 5, 19.0, 215),
    ("Target Reached", 215, 5, 19.4, 215),
]

results = []
for label, r, w, ov, tgt in scenarios:
    st = WinProbabilityState.from_overs(r, w, ov, tgt, batting_team="CSK", bowling_team="GT")
    p = model.predict(st)
    results.append({
        "Scenario": label,
        "Score": f"{r}/{w} ({ov:.1f} ov)",
        "RRR": f"{p.required_run_rate:.2f}",
        "Pressure Index": f"{p.pressure_index:.2f}",
        "CSK Win Prob (%)": f"{p.chasing_win_pct:.1f}%",
        "GT Win Prob (%)": f"{p.defending_win_pct:.1f}%",
        "Dominant Team": p.dominant_team
    })

df_scenarios = pd.DataFrame(results)
print(df_scenarios.to_string(index=False))